# Clustering tipologi stasiun

Mengevaluasi KMeans k=2..5 dengan silhouette, menolak singleton, dan menerjemahkan cluster menjadi label transit yang dapat dijelaskan.

Semua input dan output saat ini adalah prototipe sintetis. Hasil tidak boleh dianggap sebagai observasi lapangan atau rekomendasi bisnis/investasi produksi.

In [1]:
from pathlib import Path

def find_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'ml' / 'DATASET_CATALOG.md').exists():
            return candidate
    raise FileNotFoundError('Jalankan notebook dari dalam repository TCI')

ROOT = find_root()
ML_ROOT = ROOT / 'ml'
SEED = 20260911
STATUS = 'synthetic_prototype'
print(f'Project root: {ROOT}')

Project root: C:\Users\axels\Axel Documents\Documents\BINUS\Lomba\MAPID WebGIS (Top 50)\App\TCI


In [2]:

import json
import pickle
import platform
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import sklearn
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

feature_dir = ML_ROOT / "clustering"
data_path = feature_dir / "data" / "station_master.csv"
models_dir, outputs_dir = feature_dir / "models", feature_dir / "outputs"
models_dir.mkdir(parents=True, exist_ok=True)
outputs_dir.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(data_path)

numeric = ["connected_lines", "is_transit_hub", "distance_to_monas_km", "annual_passengers_2025", "daily_passengers_2025", "yoy_growth_2025_pct", "business_density_750m", "pedestrian_access_index"]
categorical = ["station_type", "line", "city_regency"]
duplicate_codes = int(df.station_code.duplicated().sum())
invalid_coordinates = int((~df.latitude.between(-7.0, -5.8) | ~df.longitude.between(105.8, 107.5)).sum())
missing_values = {column: int(value) for column, value in df[numeric + categorical].isna().sum().items() if value}
quality_passed = not (duplicate_codes or invalid_coordinates or missing_values)
quality_report = {"passed": quality_passed, "record_count": len(df), "station_count": int(df.station_code.nunique()), "duplicate_station_codes": duplicate_codes, "invalid_coordinates": invalid_coordinates, "missing_values": missing_values}
(outputs_dir / "data_quality_report.json").write_text(json.dumps(quality_report, indent=2), encoding="utf-8")
if not quality_passed:
    raise ValueError("Clustering quality gate failed; outputs are unavailable")

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric),
    ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), categorical),
])
matrix = preprocessor.fit_transform(df[numeric + categorical])
candidates = []
for k in range(2, 6):
    model = KMeans(n_clusters=k, random_state=SEED, n_init=30).fit(matrix)
    sizes = pd.Series(model.labels_).value_counts().sort_index().to_dict()
    candidates.append({"k": k, "silhouette": silhouette_score(matrix, model.labels_), "cluster_sizes": {str(key): int(value) for key, value in sizes.items()}, "has_singleton": min(sizes.values()) == 1})
valid_candidates = [candidate for candidate in candidates if not candidate["has_singleton"]]
if not valid_candidates:
    raise ValueError("All candidate clusterings contain a singleton")
selected = max(valid_candidates, key=lambda item: item["silhouette"])
model = KMeans(n_clusters=selected["k"], random_state=SEED, n_init=30).fit(matrix)
df["cluster_id"] = model.labels_

profiles = df.groupby("cluster_id").agg(
    station_count=("station_code", "size"),
    annual_passengers_mean=("annual_passengers_2025", "mean"),
    daily_passengers_mean=("daily_passengers_2025", "mean"),
    yoy_growth_mean_pct=("yoy_growth_2025_pct", "mean"),
    connected_lines_mean=("connected_lines", "mean"),
    transit_hub_share=("is_transit_hub", "mean"),
    business_density_mean=("business_density_750m", "mean"),
    pedestrian_access_mean=("pedestrian_access_index", "mean"),
    distance_to_monas_mean_km=("distance_to_monas_km", "mean"),
).reset_index()
activity = (
    profiles.annual_passengers_mean.rank(pct=True)
    + profiles.connected_lines_mean.rank(pct=True)
    + profiles.transit_hub_share.rank(pct=True)
    + profiles.business_density_mean.rank(pct=True)
    + profiles.pedestrian_access_mean.rank(pct=True)
    - profiles.distance_to_monas_mean_km.rank(pct=True)
)
ordered = profiles.assign(activity_index=activity).sort_values("activity_index").cluster_id.tolist()
label_options = {
    2: ["Aktivitas lebih rendah", "Hub aktivitas tinggi"],
    3: ["Aktivitas lebih rendah", "Koridor komuter menengah", "Hub aktivitas tinggi"],
    4: ["Aktivitas lebih rendah", "Koridor komuter menengah", "Hub berkembang", "Hub aktivitas tinggi"],
    5: ["Aktivitas lebih rendah", "Koridor lokal", "Koridor komuter menengah", "Hub berkembang", "Hub aktivitas tinggi"],
}
label_map = {int(cluster): label for cluster, label in zip(ordered, label_options[selected["k"]])}
df["typology_label"] = df.cluster_id.map(label_map)
df["source_status"] = "synthetic_prototype"
output_columns = ["station_code", "station_id", "station_name", "line", "latitude", "longitude", "cluster_id", "typology_label", "source_status"]
df[output_columns].to_csv(outputs_dir / "station_typologies.csv", index=False)

profile_records = []
for row in profiles.itertuples(index=False):
    profile_records.append({
        "cluster_id": int(row.cluster_id), "label": label_map[int(row.cluster_id)],
        "station_count": int(row.station_count),
        "annual_passengers_mean": row.annual_passengers_mean,
        "daily_passengers_mean": row.daily_passengers_mean,
        "yoy_growth_mean_pct": row.yoy_growth_mean_pct,
        "connected_lines_mean": row.connected_lines_mean,
        "transit_hub_share": row.transit_hub_share,
        "business_density_mean": row.business_density_mean,
        "pedestrian_access_mean": row.pedestrian_access_mean,
        "distance_to_monas_mean_km": row.distance_to_monas_mean_km,
    })
(outputs_dir / "cluster_profiles.json").write_text(json.dumps({"selected_k": selected["k"], "silhouette": selected["silhouette"], "profiles": profile_records, "source_status": "synthetic_prototype"}, indent=2), encoding="utf-8")
artifact = {"preprocessor": preprocessor, "model": model, "numeric_features": numeric, "categorical_features": categorical, "cluster_label_map": label_map, "source_status": "synthetic_prototype"}
with (models_dir / "typology_pipeline.pkl").open("wb") as file:
    pickle.dump(artifact, file)
(outputs_dir / "metrics.json").write_text(json.dumps({"selected_k": selected["k"], "selected_silhouette": selected["silhouette"], "candidates": candidates, "label_mapping": label_map, "source_status": "synthetic_prototype"}, indent=2), encoding="utf-8")
(outputs_dir / "feature_schema.json").write_text(json.dumps({"numeric_features": numeric, "categorical_features": categorical, "identifier_fields": ["station_code", "station_id"], "target": None}, indent=2), encoding="utf-8")
manifest = {"run_at_utc": datetime.now(timezone.utc).isoformat(), "source": str(data_path.relative_to(ROOT)), "records": len(df), "selected_k": selected["k"], "seed": SEED, "python": platform.python_version(), "sklearn": sklearn.__version__, "source_status": "synthetic_prototype", "limitations": "Typologies use synthetic transit profiles; labels are descriptive, not residential/property claims."}
(outputs_dir / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps({"quality_passed": quality_passed, "selected_k": selected["k"], "silhouette": selected["silhouette"], "model": str(models_dir / 'typology_pipeline.pkl')}, indent=2))


{
  "quality_passed": true,
  "selected_k": 2,
  "silhouette": 0.5246161109315419,
  "model": "C:\\Users\\axels\\Axel Documents\\Documents\\BINUS\\Lomba\\MAPID WebGIS (Top 50)\\App\\TCI\\ml\\clustering\\models\\typology_pipeline.pkl"
}
